# 01 - PADO-Pauli 퀵스타트

이 노트북에서 다루는 내용:

- Pauli-Propagation Surrogate(PPS)란 무엇이고, 왜 "한 번 컴파일하고 여러 번 평가"하는가?
- `Circuit` 빌더로 회로를 만들고, 관측량과 함께 컴파일해 기댓값을 얻는 방법은?
- gradient를 구하고, 파라미터 배치를 평가하고, 데이터 임베딩을 넣는 방법은?
- surrogate의 정확도·비용 균형(절단)을 조절하고, 노이즈를 추가하고, PennyLane과 교차 검증하는 방법은?

## 노트북 가이드

- 위에서 아래로 순서대로 따라가며 읽으면 됩니다. 각 섹션은 바로 실행해 볼 수 있는 단계로 구성되어 있습니다.
- 섹션 순서: 개념 → 회로 작성·draw·컴파일 → 평가 → gradient → 배치/임베딩 → 절단 → 노이즈 → 교차 검증.
- 학습 목표: 직접 구성한 회로를 컴파일하고 미분할 수 있습니다.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import torch

import padopauli
from padopauli import Circuit

use_cuda = torch.cuda.is_available()
nb_preset = "gpu" if use_cuda else "cpu"
print(f"padopauli {padopauli.__version__}")
print(f"torch {torch.__version__} | cuda available: {use_cuda} | preset: {nb_preset}")


padopauli 2.2.2
torch 2.11.0+rocm7.2 | cuda available: True | preset: gpu


## 1) 핵심 아이디어

PADO-Pauli는 양자 기댓값을 **Pauli-Propagation Surrogate(PPS)**로 계산합니다.
회로와 관측량을 **한 번** 희소 텐서 연산자 프로그램으로 컴파일합니다
(관측량을 하이젠베르크 묘사(Heisenberg picture)에서 역방향으로 전파하고, 입력 상태 |0...0⟩에서 기여하지 못하는 항을
정확히 걸러내는 zero filter를 적용합니다). 이렇게 한 번만 컴파일해 두면, 이후에는 **어떤** 파라미터 벡터에 대해서도
적은 비용으로 빠르게 평가할 수 있고 미분도 완벽히 지원됩니다.

```
회로 + 관측량  --컴파일-->  희소 텐서 프로그램  --평가-->  ⟨O⟩(θ), ∇⟨O⟩(θ)
    (한 번)                                       (여러 번, 배치로, GPU에서)
```

회로는 **`Circuit` 빌더**로 구성합니다. 각 게이트 메서드는 `self`를 반환하므로 메서드를 연달아 호출(체이닝)할 수 있습니다.

- `qc.rx(q, param_idx=i)` / `qc.rzz(q0, q1, param_idx=i)` / 일반형 `qc.rot("XXXY", [0, 1, 2, 3], param_idx=i)`:
  exp(-i θ[i]/2 · P). **같은 `param_idx`를 여러 게이트에서 재사용**할 수 있습니다(파라미터 공유).
- `qc.rx(q, embedding_idx=j)`: 회전 각도를 θ가 아니라 별도의 **데이터 벡터**에서 읽어옵니다(6절).
- `qc.h(q)`, `qc.x(q)`, `qc.s(q)`, `qc.cnot(control, target)`, `qc.cz(q0, q1)`: 엔진이 직접 처리하는 Clifford 게이트.
- `qc.depolarizing(q, px, py, pz)`, `qc.amplitude_damping(q, gamma)`: 노이즈 채널(7절).
- `qc.draw()`: 언제든 ASCII 회로도를 출력합니다. 컴파일 후에는 관측량도 함께 표시됩니다.

`Circuit`은 함수형 API(`PauliRotation` 리스트 + `compile_program`)를 감싼 고수준 인터페이스(래퍼)이며, 기존의 함수형 API도 그대로 사용할 수 있습니다.

In [2]:
n = 6

# 간단한 hardware-efficient 회로: RX 레이어 -> 얽힘 RZZ 사다리 -> RX 레이어
qc = Circuit(n)
pidx = 0
for q in range(n):
    qc.rx(q, param_idx=pidx); pidx += 1
for q in range(n - 1):
    qc.rzz(q, q + 1, param_idx=pidx); pidx += 1
qc.cnot(0, 1)
for q in range(n):
    qc.rx(q, param_idx=pidx); pidx += 1

n_params = qc.n_params
print(f"{len(qc.gates)} gates, {n_params} parameters")
qc.draw()

# 관측량: (pauli, qubits[, coeff]) 튜플. 튜플의 리스트 하나는 다항 관측량 하나를 나타냅니다
observables = [
    ("Z", [0]),                                        # <Z_0>
    ("ZZ", [0, n - 1]),                                # 양 끝 큐비트의 상관자
    [("ZZ", [q, q + 1], 1.0) for q in range(n - 1)],   # Ising 체인 에너지(단일 PauliSum)
]

18 gates, 17 parameters
q0:─RXθ₀─RZθ₆───────●───RXθ₁₁────────────
          │         │
q1:─RXθ₁──RZ──RZθ₇──X───RXθ₁₂────────────
               │
q2:─RXθ₂───────RZ──RZθ₈─RXθ₁₃────────────
                    │
q3:─RXθ₃────────────RZ───RZθ₉─RXθ₁₄──────
                          │
q4:─RXθ₄──────────────────RZ──RZθ₁₀─RXθ₁₅
                                │
q5:─RXθ₅────────────────────────RZ──RXθ₁₆


## 2) 한 번 컴파일하고 여러 번 평가

`qc.compile(observables=..., preset=...)`는 회로를 컴파일하여 생성된 프로그램을 회로 내부에 저장하는 동시에,
**해당 프로그램 객체를 반환합니다**(`CompiledProgram`: 함수형 API가 생성하는 것과 동일한 객체이므로
두 방식을 혼용할 수 있습니다). `preset`은 프로그램이 메모리에 적재되고 실행될 장치를 지정합니다:
`"cpu"`, `"gpu"`, `"hybrid"`(CPU 저장 + GPU 연산). 프리셋의 기본 dtype은 `float64`이며,
이 노트북에서는 설정이 잘 드러나도록 `dtype="float64"`를 명시적으로 지정합니다.

In [3]:
program = qc.compile(observables=observables, preset=nb_preset, dtype="float64")

theta = torch.zeros(n_params, dtype=torch.float64)
print("expvals at theta=0:", qc.expvals(theta).tolist())   # |0..0> -> Z 대각 관측량은 모두 1

theta = torch.linspace(0.1, 1.2, n_params, dtype=torch.float64)
ev = qc.expvals(theta)
print("expvals:", [f"{v:+.6f}" for v in ev.tolist()])

propagate:   0%|          | 0/18 [00:00<?, ?it/s]

[PPS Info] Propagation complete. Terms generated: 10


zero-filter:   0%|          | 0/18 [00:00<?, ?it/s]

[PPS Info] Terms retained after pruning: 10 (100.000000% of peak)


expvals at theta=0: [1.0, 1.0, 5.0]
expvals: ['+0.645242', '+0.028977', '+1.109788']


## 3) Gradient

`requires_grad=True`로 설정된 θ를 전달하고 `.backward()`를 호출합니다. 기본값인 `diff_mode="vjp"`는
직접 구현된 vector-Jacobian product(VJP)를 사용하며, `diff_mode="autograd"`는 호스트의 autograd를 통해
수반(adjoint) 그래프를 미분합니다. **두 방식이 계산하는 gradient는 동일합니다.**

In [4]:
theta = torch.linspace(0.1, 1.2, n_params, dtype=torch.float64, requires_grad=True)

loss = qc.expvals(theta)[2]          # 체인 에너지 관측량
loss.backward()                      # VJP로 gradient 계산
g_manual = theta.grad.clone()

theta.grad = None
loss = qc.expvals(theta, diff_mode="autograd")[2]
loss.backward()
g_auto = theta.grad.clone()

print("max |grad| difference (vjp vs autograd):",
      float((g_manual - g_auto).abs().max()))
print("first 5 gradient entries:", [f"{v:+.6f}" for v in g_manual[:5].tolist()])

max |grad| difference (vjp vs autograd): 5.551115123125783e-17
first 5 gradient entries: ['-0.320686', '-0.432911', '-0.195304', '-0.017616', '+0.257038']


## 4) 파라미터 배치

`qc.expvals`는 `(batch, n_params)` 형태의 행렬을 입력받아 `(batch, n_obs)`를 반환합니다. 컴파일된 프로그램 하나로
여러 파라미터 조합을 한 번에 평가할 수 있습니다(최적화기 모집단 평가나 손실 지형 탐색이 이 방식으로 동작합니다).

In [5]:
theta_batch = torch.rand(8, n_params, dtype=torch.float64)
ev_batch = qc.expvals(theta_batch)
print("theta batch", tuple(theta_batch.shape), "-> expvals", tuple(ev_batch.shape))

theta batch (8, 17) -> expvals (8, 3)


## 5) 프리셋과 절단(정확도 대 비용)

surrogate는 기본적으로 정확한 값을 계산합니다. 규모가 큰 회로에서는 컴파일 시점의 **절단(truncation)**을 통해 정확도와 계산 비용 사이의 균형을 조절할 수 있습니다.

- `qc.compile(..., max_weight=w)`: 전파 과정에서 weight > w인 Pauli 항을 제거합니다.
- `qc.compile(..., build_thetas=θ0, build_min_abs=ε)`: 기준점 θ0에서 계수의 절댓값이 ε보다 작은 항도 함께 제거합니다
  (**주의:** 프로그램이 θ0 근방에 맞춰 구성되므로 단일 지점 평가에는 적합하지만, θ0에서 멀리 벗어나는 학습 과정에는 적합하지 않습니다).

`compile()`은 프로그램 객체를 반환하므로 동일한 회로에 대해 여러 버전의 컴파일 결과를 동시에 유지할 수 있습니다.
단, `qc.expvals`는 항상 **가장 마지막에** 컴파일된 프로그램을 사용합니다. 절단 후 남은 항의 개수가 곧 계산 비용의 척도가 됩니다.

In [6]:
def nnz(prog):
    return prog.step_nnz   # 전체 스텝 행렬에 저장된 비영(non-zero) 원소 수(2.1.0: CompiledProgram.step_nnz)

theta_eval = torch.linspace(0.1, 1.2, n_params, dtype=torch.float64)
full = program.expvals(theta_eval)             # `program` = 2절에서 컴파일한 정확한 프로그램

print(f"{'max_weight':>10} | {'nnz':>6} | max |error| vs exact")
for w in [1, 2, 3]:
    p_w = qc.compile(observables=observables, preset=nb_preset, dtype="float64",
                     max_weight=w, verbosity=0)
    err = float((p_w.expvals(theta_eval) - full).abs().max())
    print(f"{w:>10} | {nnz(p_w):>6} | {err:.3e}")
print(f"{'exact':>10} | {nnz(program):>6} | 0")

max_weight |    nnz | max |error| vs exact
         1 |      1 | 1.110e+00


         2 |     50 | 4.035e-01


         3 |     73 | 3.741e-02
     exact |     85 | 0


## 6) 데이터 임베딩(양자 머신러닝)

`embedding_idx=j`가 지정된 회전 게이트는 평가 시점에 전달하는 별도의 **데이터 벡터**에서 각도를 읽어옵니다. 즉, QML 모델의
데이터 인코딩 레이어 역할을 합니다. 임베딩 입력은 최적화 대상 파라미터가 아니라 입력 데이터로 취급됩니다(단, autograd 그래프는
그대로 연결되므로 임베딩을 생성하는 앞단의 고전 레이어로도 gradient가 전달됩니다. `08_hybrid_classical_quantum.ipynb` 참조).
임베딩과 θ는 **독립적으로** 배치 처리됩니다: `qc.expvals(theta, embedding=x)`에 `(p_bs, P)` 크기의 θ와 `(e_bs, E)` 크기의 x를
전달하면 `(e_bs, p_bs, n_obs)`가 반환되어, 모든 데이터 샘플 × 모든 파라미터 조합을 한 번에 평가합니다.

In [7]:
qml_qc = Circuit(n)
for q in range(n):                    # 데이터 인코딩 레이어: 각도 = x[q]
    qml_qc.rx(q, embedding_idx=q)
pidx = 0
for q in range(n - 1):                # 학습 가능한 얽힘 게이트
    qml_qc.rzz(q, q + 1, param_idx=pidx); pidx += 1
for q in range(n):                    # 학습 가능한 회전 레이어
    qml_qc.ry(q, param_idx=pidx); pidx += 1

qml_qc.compile(observables=[("Z", [0])], preset=nb_preset, dtype="float64")

theta_b = torch.rand(3, qml_qc.n_params, dtype=torch.float64)   # 파라미터 조합 3개
x_b = torch.rand(5, qml_qc.n_embedding, dtype=torch.float64)    # 데이터 샘플 5개
out = qml_qc.expvals(theta_b, embedding=x_b)
print("theta (3, P) x data (5, E) -> output", tuple(out.shape))   # (5, 3, 1)

propagate:   0%|          | 0/17 [00:00<?, ?it/s]

[PPS Info] Propagation complete. Terms generated: 2


zero-filter:   0%|          | 0/5 [00:00<?, ?it/s]

[PPS Info] Terms retained after pruning: 2 (100.000000% of peak)
theta (3, P) x data (5, E) -> output (5, 3, 1)


## 7) 노이즈 채널

Pauli 노이즈는 surrogate 전파 과정에 정확히 반영됩니다. `qc.depolarizing(...)`이나 `qc.amplitude_damping(...)`을
일반 게이트 메서드처럼 회로 중간에 추가하기만 하면 됩니다.

In [8]:
noisy = Circuit(n)
pidx = 0
for q in range(n):
    noisy.rx(q, param_idx=pidx); pidx += 1
for q in range(n - 1):
    noisy.rzz(q, q + 1, param_idx=pidx); pidx += 1
    noisy.depolarizing(q, 5e-3, 5e-3, 5e-3)       # 각 얽힘 게이트 직후에 노이즈 추가
noisy.cnot(0, 1)
for q in range(n):
    noisy.rx(q, param_idx=pidx); pidx += 1

noisy.compile(observables=observables, preset=nb_preset, dtype="float64", verbosity=0)
print("ideal :", [f"{v:+.6f}" for v in program.expvals(theta_eval).tolist()])
print("noisy :", [f"{v:+.6f}" for v in noisy.expvals(theta_eval).tolist()])

ideal : ['+0.645242', '+0.028977', '+1.109788']
noisy : ['+0.632470', '+0.028403', '+1.069895']


## 8) PennyLane과 교차 검증

소규모 시스템(≤ 20 큐비트)에서는 컴파일된 프로그램의 결과를 동일한 회로의 PennyLane 상태벡터 시뮬레이션 결과와 비교해 검증할 수 있습니다.

In [9]:
ref = program.expvals_reference(theta_eval, max_qubits=20)
dev_max = float((program.expvals(theta_eval) - ref).abs().max())
print("PennyLane reference :", [f"{v:+.6f}" for v in ref.tolist()])
print("max |deviation|     :", f"{dev_max:.3e}")
assert dev_max < 1e-9

PennyLane reference : ['+0.645242', '+0.028977', '+1.109788']
max |deviation|     : 6.217e-15


## 다음 단계

- **`tutorial/`의 나머지 노트북**: 참조 API(02), 명시적 학습 루프(03), 배치 임베딩 입력(04),
  GPU 예산에 맞춘 프리셋 조정(05), quasi-probability 복원(06), 파라미터 배치 평가(07),
  고전-양자 하이브리드 모델(08).
- **`../examples/`**: 전체 응용 예제. 127큐비트 kicked Ising(01), H2 VQE(02), MaxCut-QAOA(03),
  SAFE ma-QAOA surrogate 사전 학습(warm-start) 워크플로(04).